# 08 · Segunda prueba de clustering: empleo, viento y agua subterránea (issue #15 · ML-03)

**Objetivo:** repetir el clustering añadiendo a las 4 variables de la primera prueba la **tasa de paro**, el **viento** y el **agua subterránea**, y comprobar si el tratamiento de los huecos de empleo condiciona el resultado.

**Variables (7):** población 2025, % de suelo industrial, distancia a autovía, distancia a terminal ferroviaria, tasa de paro, viento a 100 m y agua subterránea en mal estado cuantitativo.

**Decisiones sobre los huecos (todas modificables en las configuraciones de la sección 2):**

| Hueco | Municipios | Regla para entrenar | Motivo |
|---|---|---|---|
| Paro `ocultado_menor5` | 1.587 | Se calcula la tasa con 2,5 parados | El estado indica un valor menor que 5 (secreto estadístico del SEPE); 2,5 es el punto medio de 1 a 4 |
| Paro `blanco_fuente` | 265 | Se calcula la tasa con 1 parado | Municipios muy pequeños (población mediana de 34); la fuente no dice el valor, así que es un supuesto. Acordado con el equipo. Queda marcado en `paro_origen` |
| Agua subterránea sin masa catalogada | 722 | Media de los 5 municipios con dato más cercanos en el mapa | Elegida por el equipo: la geografía predice bien el agua (comprobación en la sección 1). Es una estimación y se mostrará como `aproximado`. Rellenar con 0 diría "buen estado", que no es lo que sabemos |
| Viento (Ceuta y Melilla) | 2 | Mediana | Como en la metodología |

El paro rellenado es **solo para entrenar**: la app muestra el dato observado y su estado (en `estado_app` ambos estados de paro son `pendiente`). El agua estimada se propone mostrarla como `aproximado` (fichero `agua_estimada_vecinos_v01.csv`, pendiente de que Elena lo incorpore al dataset).

**Prueba de sensibilidad:** la misma lista con `eliminar_filas` en el paro (se pierden los 1.852 municipios sin dato) y comparación de los grupos de los municipios comunes.

In [ ]:
import json, subprocess, sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)

SEMILLA = 42
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data').is_dir() and (p / 'notebooks').is_dir())
DATA = ROOT / 'data/processed/sitewise_v03'
GENERADOR = ROOT / 'data/processed/generadorMatrices'
SALIDA = ROOT / 'data/processed/sitewise_ml'
MATRICES = SALIDA / 'mlflow_matrices_2'
for f in [DATA / 'dataset_municipal.csv', DATA / 'valores.csv', GENERADOR / 'generar_matriz.py', SALIDA / 'clusters_kmeans_v01.csv']:
    assert f.exists(), f'Falta {f.relative_to(ROOT)}'

MLRUNS = Path(os.environ.get('SITEWISE_MLRUNS', ROOT / 'mlruns'))
MLRUNS.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri('sqlite:///' + (MLRUNS / 'mlflow.db').as_posix())
NOMBRE_EXPERIMENTO = 'sitewise-clustering-segunda-prueba'
if mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO) is None:
    mlflow.create_experiment(NOMBRE_EXPERIMENTO, artifact_location=(MLRUNS / 'artifacts_segunda_prueba').as_uri())
mlflow.set_experiment(NOMBRE_EXPERIMENTO)
pd.set_option('display.width', 220)
print('MLflow', mlflow.__version__, '| experimento:', NOMBRE_EXPERIMENTO)

## 1. Calcular la tasa de paro en un CSV derivado

El generador no calcula tasas, así que se prepara una copia del dataset con dos columnas nuevas. **El dataset original no se toca.**

- `paro_por_100_hab`: paro registrado por cada 100 habitantes (población total, no población activa). Para `ocultado_menor5` se usa 2,5 parados y para `blanco_fuente`, 1 parado. Así la variable no tiene huecos.
- `paro_por_100_hab_sin_ocultos`: solo los valores publicados (los 1.852 huecos quedan vacíos), para la prueba de sensibilidad.
- `paro_origen`: de dónde viene cada valor, para poder rastrearlo.
- `agua_vecinos_pct` y `agua_origen`: el agua observada y, para los 722 sin masa catalogada, la media de los 5 municipios con dato más cercanos (`estimado_vecinos`). Es la opción elegida por el equipo; en la app se mostrará como `aproximado`.
- `agua_sin_dato`: 1 si el municipio no tiene masa de agua subterránea catalogada (722) y 0 si tiene dato. Sirve para la variante que rellena con 0 pero avisa al algoritmo de que ese 0 no es una medición.

In [ ]:
d = pd.read_csv(DATA / 'dataset_municipal.csv', dtype={'cod_ine': str})
v = pd.read_csv(DATA / 'valores.csv', dtype={'cod_ine': str})
estado = (v[v['id_indicador'] == 'paro_total'].set_index('cod_ine')['estado']
          .replace({'agregado_fusion_documentada': 'publicado'}).rename('paro_origen'))
d = d.merge(estado, left_on='cod_ine', right_index=True, how='left')
assert d['paro_origen'].notna().all() and len(d) == 8132

tasa = d['paro_total'] / d['poblacion_2025'] * 100
d['paro_por_100_hab_sin_ocultos'] = tasa.where(d['paro_origen'] == 'publicado')
oculto = d['paro_origen'] == 'ocultado_menor5'
blanco = d['paro_origen'] == 'blanco_fuente'
d['paro_por_100_hab'] = (d['paro_por_100_hab_sin_ocultos']
                         .mask(oculto, 2.5 / d['poblacion_2025'] * 100)
                         .mask(blanco, 1 / d['poblacion_2025'] * 100))
assert d['paro_por_100_hab'].notna().all()

# Marca de "sin dato de agua": 1 si el municipio no tiene masa subterránea catalogada (722), 0 si tiene dato.
d['agua_sin_dato'] = d['agua_subterranea_mal_estado_cuantitativo_pct'].isna().astype(int)

# Agua estimada para los 722 sin masa catalogada: media de los 5 municipios con dato más cercanos en el mapa.
from sklearn.neighbors import BallTree
COL_AGUA = 'agua_subterranea_mal_estado_cuantitativo_pct'
K_VECINOS = 5
tiene = d[COL_AGUA].notna().to_numpy()
coords = np.radians(d[['lat', 'lon']].to_numpy())
arbol = BallTree(coords[tiene], metric='haversine')
dist, idx = arbol.query(coords[~tiene], k=K_VECINOS)
d['agua_vecinos_pct'] = d[COL_AGUA]
d.loc[~tiene, 'agua_vecinos_pct'] = d.loc[tiene, COL_AGUA].to_numpy()[idx].mean(axis=1)
d['agua_origen'] = np.where(tiene, 'observado', 'estimado_vecinos')
assert d['agua_vecinos_pct'].notna().all()

CSV_DERIVADO = SALIDA / 'dataset_municipal_tasa_paro.csv'
d.to_csv(CSV_DERIVADO, index=False)

# Fichero candidato para el dataset: los valores estimados, con su método, para que Elena decida cómo entran.
propuesta = pd.DataFrame({
    'cod_ine': d.loc[~tiene, 'cod_ine'].to_numpy(),
    'valor_estimado_pct': d.loc[~tiene, 'agua_vecinos_pct'].round(2).to_numpy(),
    'k_vecinos': K_VECINOS,
    'distancia_media_km': (dist.mean(axis=1) * 6371).round(1),
    'distancia_maxima_km': (dist.max(axis=1) * 6371).round(1),
    'estado_propuesto': 'aproximado',
    'metodo': f'Media del indicador en los {K_VECINOS} municipios con dato más cercanos (distancia entre coordenadas). Solo para municipios sin masa catalogada.'})
propuesta.to_csv(SALIDA / 'agua_estimada_vecinos_v01.csv', index=False)
print('Agua estimada para', len(propuesta), 'municipios (media', propuesta['valor_estimado_pct'].mean().round(1), '%); distancia mediana a los vecinos',
      propuesta['distancia_media_km'].median(), 'km')
print(d['paro_origen'].value_counts().to_string())
print('agua_sin_dato:', d['agua_sin_dato'].value_counts().to_dict())
print()
print(d.groupby('paro_origen')[['paro_por_100_hab']].describe().droplevel(0, axis=1)[['count', 'min', '50%', 'max']].round(2).to_string())

### Comprobación: ¿predice bien la geografía el agua subterránea?

Con los municipios que sí tienen dato se borra el valor de cada uno y se estima de varias maneras. Error medio en puntos porcentuales (más bajo = mejor).

In [ ]:
from sklearn.neighbors import NearestNeighbors
con = d[COL_AGUA].notna().to_numpy()
y = d.loc[con, COL_AGUA].to_numpy()
c_con = coords[con]
_, i_geo = BallTree(c_con, metric='haversine').query(c_con, k=K_VECINOS + 1)
p_geo = y[i_geo[:, 1:]].mean(axis=1)
Xv = d.loc[con, ['poblacion_2025', 'suelo_industrial_cartografiado_pct', 'dist_recta_eje_autovia_km', 'dist_recta_ttm_km',
                 'paro_por_100_hab', 'viento_100m_media_m_s']].apply(lambda s: (s.fillna(s.median()) - s.fillna(s.median()).mean()) / s.fillna(s.median()).std()).to_numpy()
_, i_var = NearestNeighbors(n_neighbors=K_VECINOS + 1).fit(Xv).kneighbors(Xv)
p_var = y[i_var[:, 1:]].mean(axis=1)
comparacion = pd.DataFrame({'error_medio': {
    'poner 0 (relleno anterior)': np.abs(y).mean(),
    'media de todos los municipios': np.abs(y - y.mean()).mean(),
    '5 municipios parecidos en las otras 6 variables': np.abs(y - p_var).mean(),
    '5 municipios más cercanos en el mapa': np.abs(y - p_geo).mean()}}).round(1)
print(comparacion.to_string())
print('Correlación con el valor real (vecinos del mapa):', round(float(np.corrcoef(p_geo, y)[0, 1]), 2))

## 2. Configuraciones del generador

Se parte de la configuración de 4 variables de Elena y se añaden las nuevas. Cinco variantes:

- **`empleo_viento_agua_vecinos`**: la **configuración elegida** (7 variables; el agua de los 722 sin masa catalogada se estima con los 5 municipios con dato más cercanos).
- **`empleo_viento_agua`**: las 7 variables con el agua sin dato rellenada con 0 (opción anterior, para comparar).
- **`empleo_viento`**: sin agua subterránea, para ver cuánto influye esa variable y su relleno con 0.
- **`empleo_viento_agua_marca`**: como la principal, pero añadiendo la marca `agua_sin_dato` (0/1) como octava variable, para que los 722 municipios sin masa catalogada no se confundan con los que tienen un 0 medido.
- **`sensib_eliminar_filas`**: la principal, pero eliminando los municipios sin dato de paro.

In [ ]:
base = json.loads((GENERADOR / 'config_4_variables.json').read_text(encoding='utf-8-sig'))
contexto = base['contexto'] + ['paro_origen']

paro = {'ausentes': 'error', 'transformacion': 'ninguna', 'recorte': [0.01, 0.99]}  # sin huecos: si apareciera alguno, el generador falla
viento = {'ausentes': 'mediana', 'transformacion': 'ninguna'}
agua = {'ausentes': 'constante', 'valor': 0, 'transformacion': 'ninguna'}

def config(extra):
    return {**base, 'contexto': contexto, 'variables': {**base['variables'], **extra}}

CONFIGS = {
    'empleo_viento_agua': config({'paro_por_100_hab': paro, 'viento_100m_media_m_s': viento,
                                  'agua_subterranea_mal_estado_cuantitativo_pct': agua}),
    'empleo_viento_agua_vecinos': config({'paro_por_100_hab': paro, 'viento_100m_media_m_s': viento,
                                          'agua_vecinos_pct': {'ausentes': 'error', 'transformacion': 'ninguna'}}),
    'empleo_viento': config({'paro_por_100_hab': paro, 'viento_100m_media_m_s': viento}),
    'empleo_viento_agua_marca': config({'paro_por_100_hab': paro, 'viento_100m_media_m_s': viento,
                                        'agua_subterranea_mal_estado_cuantitativo_pct': agua,
                                        'agua_sin_dato': {'ausentes': 'error', 'transformacion': 'ninguna'}}),
    'sensib_eliminar_filas': config({'paro_por_100_hab_sin_ocultos': {**paro, 'ausentes': 'eliminar_filas'},
                                     'viento_100m_media_m_s': viento,
                                     'agua_subterranea_mal_estado_cuantitativo_pct': agua}),
}
MATRICES.mkdir(parents=True, exist_ok=True)

import hashlib

def generar(nombre, cfg):
    # El nombre de la carpeta lleva una huella del CSV de entrada y de la configuración:
    # si cambia cualquiera de los dos, se genera una matriz nueva en vez de reutilizar una antigua.
    huella = hashlib.sha256(CSV_DERIVADO.read_bytes() + json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:8]
    carpeta = MATRICES / f'{nombre}_{huella}'
    if not carpeta.exists():
        ruta = MATRICES / f'config_{nombre}_{huella}.json'
        ruta.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')
        res = subprocess.run([sys.executable, str(GENERADOR / 'generar_matriz.py'), '--dataset', str(CSV_DERIVADO),
                              '--config', str(ruta), '--salida', str(carpeta)], capture_output=True, text=True)
        assert res.returncode == 0, res.stderr
        print(f'Matriz {nombre} generada en', carpeta.relative_to(ROOT))
    return carpeta

carpetas = {n: generar(n, c) for n, c in CONFIGS.items()}
informes = {n: json.loads((c / 'informe.json').read_text(encoding='utf-8')) for n, c in carpetas.items()}
for n, i in informes.items():
    imp = {k: x for k, x in i['valores_imputados'].items() if x}
    print(f"{n}: filas {i['filas_matriz']} (excluidas {i['filas_excluidas']}) | PCA: {i['pca']['varianza_acumulada']:.3f} en {len(i['pca']['varianza_explicada'])} comp. | imputados: {imp}")

## 3. Cargar las matrices con PCA

Se usa `matriz_pca.csv` de cada variante, con `cod_ine` como texto y fuera del cálculo.

In [ ]:
def cargar(nombre):
    t = pd.read_csv(carpetas[nombre] / 'matriz_pca.csv', dtype={'cod_ine': str})
    return t['cod_ine'].copy(), t.drop(columns='cod_ine')

COD, MATRIZ = {}, {}
for n in CONFIGS:
    COD[n], MATRIZ[n] = cargar(n)
print({n: m.shape for n, m in MATRIZ.items()})

def parametros_comunes(nombre):
    i = informes[nombre]
    return {'configuracion': nombre, 'variables': ', '.join(i['configuracion']['variables']),
            'usa_pca': True, 'n_dimensiones': MATRIZ[nombre].shape[1],
            'varianza_pca': round(i['pca']['varianza_acumulada'], 4), 'filas': i['filas_matriz'],
            'filas_excluidas': i['filas_excluidas'], 'sha256_dataset': i['sha256_dataset'][:12]}

## 4. K-Means con K de 2 a 10

Se registra cada prueba en MLflow, igual que en el notebook 07.

In [ ]:
filas = []
for nombre in ['empleo_viento_agua_vecinos', 'empleo_viento_agua', 'empleo_viento', 'empleo_viento_agua_marca']:
    X = MATRIZ[nombre]
    for k in range(2, 11):
        km = KMeans(n_clusters=k, n_init=20, random_state=SEMILLA).fit(X)
        m = {'inercia': km.inertia_, 'silhouette': silhouette_score(X, km.labels_),
             'davies_bouldin': davies_bouldin_score(X, km.labels_),
             'calinski_harabasz': calinski_harabasz_score(X, km.labels_),
             'grupo_menor_pct': np.bincount(km.labels_).min() / len(X) * 100}
        with mlflow.start_run(run_name=f'kmeans_{nombre}_k{k}'):
            mlflow.set_tags({'algoritmo': 'kmeans', 'tipo': 'barrido_k'})
            mlflow.log_params({**parametros_comunes(nombre), 'k': k, 'semilla': SEMILLA, 'n_init': 20})
            mlflow.log_metrics(m)
            mlflow.log_artifact(str(carpetas[nombre].parent / f'config_{carpetas[nombre].name}.json'))
        filas.append({'config': nombre, 'k': k, **m})
barrido = pd.DataFrame(filas)
print('Silhouette por K:'); print(barrido.pivot(index='k', columns='config', values='silhouette').round(3).to_string())

fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, t) in zip(ejes, [('inercia', 'Inercia (codo)'), ('silhouette', 'Silhouette (más alto = mejor)'),
                                ('davies_bouldin', 'Davies-Bouldin (más bajo = mejor)')]):
    for n, g in barrido.groupby('config'):
        ax.plot(g['k'], g[col], marker='o', label=n)
    ax.set_title(t, fontsize=10); ax.set_xlabel('K'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Prueba de sensibilidad: ¿importa cómo se tratan los huecos de empleo?

Se compara, con K=5, la prueba principal con la que elimina los municipios sin dato de paro. Solo se pueden comparar los municipios que están en las dos (los 6.280 con paro publicado). Se compara también con los grupos de la primera prueba (4 variables) para ver cuánto cambia el perfil al añadir variables.

In [ ]:
K_ELEGIDO = 5
etq = {n: pd.Series(KMeans(n_clusters=K_ELEGIDO, n_init=20, random_state=SEMILLA).fit_predict(MATRIZ[n]), index=COD[n].values)
       for n in CONFIGS}
primera = pd.read_csv(SALIDA / 'clusters_kmeans_v01.csv', dtype={'cod_ine': str}).set_index('cod_ine')['grupo_kmeans']

def ari(a, b):
    comunes = a.index.intersection(b.index)
    return adjusted_rand_score(a.loc[comunes], b.loc[comunes]), len(comunes)

res = {}
res['vecinos_vs_cero'] = ari(etq['empleo_viento_agua_vecinos'], etq['empleo_viento_agua'])
res['vecinos_vs_sin_agua'] = ari(etq['empleo_viento_agua_vecinos'], etq['empleo_viento'])
res['vecinos_vs_primera_prueba'] = ari(etq['empleo_viento_agua_vecinos'], primera)
res['principal_vs_eliminar_filas'] = ari(etq['empleo_viento_agua'], etq['sensib_eliminar_filas'])
res['principal_vs_sin_agua'] = ari(etq['empleo_viento_agua'], etq['empleo_viento'])
res['principal_vs_primera_prueba'] = ari(etq['empleo_viento_agua'], primera)
res['principal_vs_agua_marca'] = ari(etq['empleo_viento_agua'], etq['empleo_viento_agua_marca'])
res['sin_agua_vs_agua_marca'] = ari(etq['empleo_viento'], etq['empleo_viento_agua_marca'])
res['sin_agua_vs_primera_prueba'] = ari(etq['empleo_viento'], primera)
tabla = pd.DataFrame(res, index=['ARI', 'municipios_comparados']).T
print(tabla.round(3).to_string())

with mlflow.start_run(run_name='sensibilidad_k5'):
    mlflow.set_tags({'algoritmo': 'kmeans', 'tipo': 'sensibilidad'})
    mlflow.log_params({'k': K_ELEGIDO, 'semilla': SEMILLA})
    mlflow.log_metrics({f'ari_{k}': v[0] for k, v in res.items()})
    mlflow.log_metrics({f'n_{k}': v[1] for k, v in res.items()})

## 6. Estabilidad de K=5 con 5 semillas

In [ ]:
SEMILLAS = [42, 7, 123, 2024, 99]
est = []
for nombre in CONFIGS:
    X = MATRIZ[nombre]
    ref = KMeans(n_clusters=K_ELEGIDO, n_init=20, random_state=SEMILLAS[0]).fit_predict(X)
    aris = [adjusted_rand_score(ref, KMeans(n_clusters=K_ELEGIDO, n_init=20, random_state=s).fit_predict(X)) for s in SEMILLAS[1:]]
    with mlflow.start_run(run_name=f'estabilidad_{nombre}_k{K_ELEGIDO}'):
        mlflow.set_tags({'algoritmo': 'kmeans', 'tipo': 'estabilidad'})
        mlflow.log_params({**parametros_comunes(nombre), 'k': K_ELEGIDO, 'semillas': str(SEMILLAS)})
        mlflow.log_metrics({'ari_medio': float(np.mean(aris)), 'ari_minimo': float(np.min(aris))})
    est.append({'config': nombre, 'ari_medio': np.mean(aris), 'ari_minimo': np.min(aris)})
print(pd.DataFrame(est).round(4).to_string(index=False))

## 7. Perfiles de la prueba principal (K=5)

Medianas de las variables originales por grupo, ordenados de mayor a menor población. La última parte muestra qué parte de cada grupo tiene el paro oculto o en blanco: si los valores calculados con una regla se concentran en un grupo, ese perfil depende más del relleno.

In [ ]:
def perfil_de(n):
    orig = pd.read_csv(carpetas[n] / 'matriz_original.csv', dtype={'cod_ine': str}).set_index('cod_ine')
    orig['grupo'] = etq[n].reindex(orig.index).to_numpy()
    ctx = pd.read_csv(carpetas[n] / 'municipios.csv', dtype={'cod_ine': str}).set_index('cod_ine')
    orig['paro_origen'] = ctx['paro_origen']
    dd = d.set_index('cod_ine')
    orig['paro_por_100_hab'] = dd['paro_por_100_hab'].reindex(orig.index)
    orig['agua_original'] = dd['agua_subterranea_mal_estado_cuantitativo_pct'].reindex(orig.index)
    g = orig.groupby('grupo')
    cols = ['poblacion_2025', 'suelo_industrial_cartografiado_pct', 'dist_recta_eje_autovia_km', 'dist_recta_ttm_km',
            'paro_por_100_hab', 'viento_100m_media_m_s']
    nombres = ['pob', 'suelo_ind_%', 'autovia_km', 'ttm_km', 'paro/100hab', 'viento']
    perfil = g[cols].median()
    perfil.columns = nombres
    perfil.insert(0, 'n', g.size())
    for c_agua in ['agua_subterranea_mal_estado_cuantitativo_pct', 'agua_vecinos_pct']:
        if c_agua in orig.columns:
            perfil['agua_mal_%'] = g[c_agua].median()
    perfil['%_paro_oculto'] = g['paro_origen'].apply(lambda s: (s == 'ocultado_menor5').mean() * 100)
    perfil['%_paro_blanco'] = g['paro_origen'].apply(lambda s: (s == 'blanco_fuente').mean() * 100)
    perfil['%_sin_agua'] = g['agua_original'].apply(lambda s: s.isna().mean() * 100)
    return perfil.sort_values('pob', ascending=False).round(2)

print('ELEGIDA (7 variables, agua estimada con vecinos)'); print(perfil_de('empleo_viento_agua_vecinos').to_string())
print(); print('7 variables, agua rellenada con 0'); print(perfil_de('empleo_viento_agua').to_string())

### Variante con la marca `agua_sin_dato`

Si la marca separa a los 722 municipios sin masa catalogada, deberían concentrarse en un grupo (`%_sin_agua` alto en uno solo). Si no, la marca no aporta información propia.

In [ ]:
print('AGUA + MARCA (8 variables)'); print(perfil_de('empleo_viento_agua_marca').to_string())

### Variante sin agua (6 variables) frente a la primera prueba (4 variables)

Perfiles de la lista sin agua, para compararlos con los de las 4 variables (cuaderno 06) y ver qué cambia al añadir solo empleo y viento. Las columnas de agua solo sirven para ver cómo caen los municipios sin dato dentro de estos grupos.

In [ ]:
print('SIN AGUA (6 variables)'); print(perfil_de('empleo_viento').to_string())
print()
pri = pd.read_csv(SALIDA / 'perfiles_kmeans_v01.csv')
print('PRIMERA PRUEBA (4 variables)'); print(pri.round(2).to_string(index=False))

## 8. Resumen de las ejecuciones de esta prueba

In [ ]:
ejecuciones = mlflow.search_runs(experiment_names=[NOMBRE_EXPERIMENTO], order_by=['start_time ASC'])
cols = ['tags.mlflow.runName', 'tags.tipo', 'params.k', 'metrics.silhouette', 'metrics.davies_bouldin',
        'metrics.ari_medio', 'metrics.ari_principal_vs_eliminar_filas', 'metrics.ari_principal_vs_sin_agua',
        'metrics.ari_principal_vs_primera_prueba', 'metrics.ari_vecinos_vs_cero', 'metrics.ari_principal_vs_agua_marca',
        'metrics.ari_sin_agua_vs_agua_marca', 'metrics.ari_sin_agua_vs_primera_prueba']
resumen = ejecuciones[[c for c in cols if c in ejecuciones.columns]].rename(columns=lambda c: c.split('.', 1)[-1])
print(len(resumen), 'ejecuciones registradas en este experimento')
print(resumen.round(3).to_string(index=False))